# Example Literature Project

## Helper Functions

In [1]:
import json
import textwrap
from pathlib import Path
from time import perf_counter

from kilin.conversations import ConversationStore
from kilin.rag import RAGSession


def _print_wrapped(label, value, width=79):
    """Print text with a fixed display width, preserving paragraphs."""
    print(f"{label}:")
    for line in str(value).splitlines() or [""]:
        if not line.strip():
            print()
        else:
            print(
                textwrap.fill(
                    line,
                    width=width,
                    break_long_words=False,
                    break_on_hyphens=False,
                )
            )
    print("")


def create_conversation(project_dir):
    """Create an independent conversation and its active RAG session.

    Parameters
    ----------
    project_dir : str or Path
        Existing KíLín project directory.

    Returns
    -------
    tuple
        Conversation store, conversation record, and active session.
        Close the session after the last question.

    """
    store = ConversationStore(Path(project_dir))
    conversation = store.create()
    path = store.state_path(conversation)
    session = RAGSession(
        Path(project_dir),
        conversation_path=path,
        router_model="Qwen/Qwen3-4B-MLX-4bit",
    )
    session.use_chat_model("lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly")
    print(f"Conversation: {path}")
    print("Answering model:", session.chat_session.model_id)
    print("Router model:", session.router_model)
    return store, conversation, session

/Users/giovannivolpe/Documents/GitHub/KiLin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### GUI-equivalent decision callbacks

In [2]:
def _show_gui_choice(question, options, selected):
    """Print the offered options and the scripted GUI selection.

    Parameters
    ----------
    question : str
        Tool question shown by the desktop GUI.
    options : tuple of str
        Choices offered by the tool.
    selected : int or None
        Chosen zero-based index, or None for cancellation.

    """
    print(f"GUI choice requested: {question}")
    for index, option in enumerate(options):
        marker = "*" if index == selected else " "
        print(f"  {marker} {index}: {option}")
    if selected is None:
        print("  Decision: cancel (no scripted choice matched)")
    else:
        print(f"  Decision: option {selected}")


def choose_first_20(question, options):
    """Choose the GUI's first-20-items chat display, when offered.

    Parameters
    ----------
    question : str
        Tool choice prompt.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Matching option index, or None when not offered.

    """
    selected = next(
        (i for i, value in enumerate(options)
         if value == "Show the first 20 here"),
        None,
    )
    _show_gui_choice(question, options, selected)
    return selected


def choose_save_all(question, options):
    """Choose the GUI option to save all matches as Markdown.

    Parameters
    ----------
    question : str
        Tool choice prompt.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Export option index, or None when not offered.

    """
    selected = next(
        (
            i for i, value in enumerate(options)
            if value.startswith("Save all ")
            and value.endswith(" to outputs/ as Markdown")
        ),
        None,
    )
    _show_gui_choice(question, options, selected)
    return selected


def choose_gui_default(question, options):
    """Use a documented automatic selection for common GUI choices.

    Long lists use the GUI's 20-item display. An ambiguous document
    question chooses the first offered paper, as if the user clicked the
    first GUI option. Every decision is printed; other choices cancel.

    Parameters
    ----------
    question : str
        Tool question.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Selected option index, or None to cancel.

    """
    selected = next(
        (i for i, value in enumerate(options)
         if value == "Show the first 20 here"),
        None,
    )
    if selected is None and options and (
        "which paper" in question.casefold()
        or "which document" in question.casefold()
    ):
        selected = 0
    _show_gui_choice(question, options, selected)
    return selected


def choose_document_named(filename):
    """Create a callback that selects one named file in a GUI prompt.

    Parameters
    ----------
    filename : str
        Filename to select, including its extension.

    Returns
    -------
    callable
        Callback with the GUI's ``(question, options)`` signature.

    """
    def choose(question, options):
        """Select the requested file from the currently offered options."""
        selected = next(
            (
                index for index, value in enumerate(options)
                if value.split(" (", 1)[0].casefold()
                == filename.casefold()
            ),
            None,
        )
        _show_gui_choice(question, options, selected)
        return selected

    return choose


def _confirm_gui_plan(plan, selection):
    """Report a GUI-equivalent plan decision, disallowing web access.

    Parameters
    ----------
    plan : ToolPlan
        Plan offered by the tool.
    selection : str
        Intended choice: continue, quick, or cancel.

    Returns
    -------
    str
        Applied choice, with network requests always cancelled.

    """
    chosen = (
        "cancel" if getattr(plan, "operation", "model") == "network"
        else selection
    )
    print(f"GUI plan: {plan.tool} | {plan.summary}")
    print(
        f"  Items: {plan.items}, batches: {plan.batches_to_run}, "
        f"quick items: {plan.quick_items}"
    )
    print(f"  Decision: {chosen}")
    return chosen


def confirm_gui_continue(plan):
    """Simulate clicking Continue, but never approve network lookup."""
    return _confirm_gui_plan(plan, "continue")


def confirm_gui_quick(plan):
    """Simulate clicking Quick, but never approve network lookup."""
    return _confirm_gui_plan(plan, "quick")


def confirm_gui_cancel(plan):
    """Simulate clicking Cancel on a tool's proposed plan."""
    return _confirm_gui_plan(plan, "cancel")



### Question and trace inspection

In [3]:
def ask_question(
    session,
    question,
    *,
    choice_callback=choose_gui_default,
    confirm_callback=confirm_gui_continue,
):
    """Ask the active model and print a width-limited exchange.

    Parameters
    ----------
    session : RAGSession
        Active conversation session.
    question : str
        User question.
    choice_callback : callable or None, optional
        Select a GUI tool option. The default displays the first 20
        matches or selects the first offered paper. None deliberately
        uses the noninteractive fallback.
    confirm_callback : callable or None, optional
        Select Continue, Quick, or Cancel for an expensive tool plan.
        The default selects Continue; network plans always cancel.

    Returns
    -------
    str
        Assistant response.

    """
    print("\n" + "=" * 79)
    _print_wrapped("USER", question)
    start = perf_counter()
    try:
        answer = session.ask(
            question,
            retrieval_count=10,
            choice_callback=choice_callback,
            confirm_callback=confirm_callback,
        )
    except Exception:
        print(f"Elapsed before failure: {perf_counter() - start:.3f} s")
        raise
    _print_wrapped("ASSISTANT", answer)
    if session.turns:
        parameters = session.turns[-1].parameters
        tool = parameters.get("tool", {})
        print(f"Resolved intent: {parameters.get('intent', 'unknown')}")
        if isinstance(tool, dict) and tool.get("name"):
            print(f"Executed tool: {tool['name']}")
    print(f"Wall time: {perf_counter() - start:.3f} s")
    return answer


def _print_retrieved_chunks(stages, width=79):
    """Display retrieved passages recorded in the native trace.

    Parameters
    ----------
    stages : dict
        Stages of one recorded conversation turn.
    width : int, optional
        Maximum display width.

    """
    retrieval = stages.get("03_retrieval", {})
    operations = retrieval.get("operations", {})

    print("\nRETRIEVED CHUNKS")

    found = 0

    def visit(value, path):
        nonlocal found

        if isinstance(value, dict):
            text = next(
                (
                    value[key]
                    for key in (
                        "text",
                        "content",
                        "chunk_text",
                        "passage",
                        "snippet",
                    )
                    if isinstance(value.get(key), str) and value[key].strip()
                ),
                None,
            )

            if text is not None:
                found += 1
                print(f"\n  CHUNK {found}")
                print(f"  Trace location: {path}")

                for key, item in value.items():
                    if key in (
                        "text",
                        "content",
                        "chunk_text",
                        "passage",
                        "snippet",
                    ):
                        continue

                    if isinstance(item, (str, int, float, bool)):
                        _print_wrapped(f"  {key}", item, width)

                _print_wrapped("  PASSAGE", text, width)
                return

            for key, child in value.items():
                visit(child, f"{path}/{key}")

        elif isinstance(value, list):
            for index, child in enumerate(value):
                visit(child, f"{path}/{index}")

    for name, operation in operations.items():
        if "retrieval" not in name.lower():
            continue

        visit(operation.get("details", {}), name)

    if not found:
        print("  No passage text found in the retrieval trace.")


def inspect_conversation(
    conversation_path,
    turn_number=None,
    show_chunks=True,
):
    """Display routing, retrieval, model calls, tools, and trace timings.

    Parameters
    ----------
    conversation_path : str or Path
        Native KíLín trace JSON file.
    turn_number : int, optional
        One-based turn index, or None for all turns.
    show_chunks : bool, optional
        Whether to display passages recorded in the retrieval trace.
        Defaults to True.

    Returns
    -------
    dict
        Unmodified native trace record.

    """
    with Path(conversation_path).open(encoding="utf-8") as file:
        trace = json.load(file)

    turns = list(trace["sessions"]["05_conversations"]["turns"].items())
    if turn_number is not None:
        if not 1 <= turn_number <= len(turns):
            raise ValueError("Invalid turn number.")
        turns = [turns[turn_number - 1]]

    def operation(stages, stage_name, operation_name):
        """Find a stage operation even if its sequence number changed."""
        operations = stages.get(stage_name, {}).get("operations", {})
        exact = operations.get(operation_name)
        if exact is not None:
            return exact
        suffix = "_" + operation_name.split("_", 1)[1]
        return next(
            (entry for key, entry in operations.items()
             if key.endswith(suffix)),
            {},
        )

    def show_fields(heading, data, keys):
        print(f"\n{heading}")
        for key in keys:
            if key in data:
                _print_wrapped(f"  {key}", data[key])

    for turn_id, turn in turns:
        stages = turn.get("stages", {})
        print("\n" + "=" * 79)
        print(f"TURN: {turn_id}")
        print("=" * 79)

        question = operation(stages, "01_question", "01_turn")
        response = operation(
            stages, "01_question", "09_conversation_turn_representation"
        )
        show_fields("EXCHANGE", question.get("details", {}), ["question"])
        show_fields("ANSWER", response.get("details", {}), ["summary"])

        decision = operation(
            stages, "01_question", "05_intent_classification_decided"
        ).get("details", {})
        show_fields(
            "INTENT",
            decision.get("classification", {}),
            ["intent", "standalone", "refers_back", "related_turn_ids"],
        )
        show_fields("ROUTING", decision, ["shortcut", "fallback"])
        show_fields(
            "RESOLVED POLICY",
            decision.get("resolved", {}),
            ["history_policy", "retrieval_mode", "retrieval_strategy"],
        )
        settings = operation(
            stages, "01_question", "02_turn_settings_snapshot"
        ).get("details", {})
        show_fields(
            "EFFECTIVE SETTINGS",
            settings,
            [
                "project_type_id",
                "backend",
                "chat_model",
                "router_model",
                "thinking",
                "temperature",
                "max_new_tokens",
                "retrieval_count",
            ],
        )
        retrieval = operation(stages, "03_retrieval", "02_retrieval_result")
        show_fields(
            "RETRIEVAL",
            retrieval.get("details", {}),
            ["mode", "strategy", "evidence_count", "duration_ms"],
        )
        if show_chunks:
            _print_retrieved_chunks(stages)

        calls = []
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                if name == "01_llm_call_input" or name == "02_llm_call_output":
                    calls.append((stage_name, name, entry))
                elif "tool" in name.lower():
                    print(f"\nTOOL OPERATION: {stage_name}/{name}")
                    show_fields(
                        "  DETAILS",
                        entry.get("details", {}),
                        ["tool", "tool_name", "status", "duration_ms"],
                    )
        for stage_name, name, entry in calls:
            details = entry.get("details", {})
            show_fields(
                f"MODEL CALL: {stage_name}/{name}",
                details,
                [
                    "call_id",
                    "model_id",
                    "backend",
                    "device",
                    "thinking",
                    "input_token_count",
                    "prompt_token_count",
                    "output_token_count",
                    "finish_reason",
                    "reply_duration_ms",
                ],
            )
            show_fields(
                "GENERATION SETTINGS",
                details.get("generation", {}),
                ["temperature", "max_new_tokens", "thinking"],
            )

        print("\nOPERATION TIMINGS (ms)")
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                timing = entry.get("timing", {})
                duration = timing.get("operation_duration_ms")
                if duration is not None:
                    print(f"  {stage_name}/{name}: {duration:.3f}")

    return trace




### Intent and tool verification helpers

In [4]:
# Intent coverage is based on saved turn metadata, not on the wording of
# the test question or the route we expected the model to choose.
intent_checks = []


def _intent_definitions():
    """Read the active repository's literature intent definitions."""
    path = repo_root / "kilin" / "project_types" / "literature.json"
    with path.open(encoding="utf-8") as file:
        return json.load(file)["intents"]


# Fields demonstrating that a tool progressed beyond initial selection.
# They are deliberately not assertions about scientific correctness.
_TOOL_RESULT_FIELDS = {
    "project_inventory": ("files",),
    "scan_and_list": ("total_mentions", "listed"),
    "show_source": ("turn", "passages"),
    "filter_results": ("turn", "items"),
    "paper_overview": ("paper",),
    "suggest_reference": ("statement", "papers_checked"),
    "find_passages": ("topic", "choice"),
    "find_fact": ("choice",),
    "batch_review": ("document", "choice"),
    "batch_classify": ("categories", "choice"),
    "reference_check": ("paper",),
    "generate_paper_summary": ("template", "saved"),
}


def check_intent(
    session,
    expected_intent,
    question,
    *,
    choice_callback=choose_gui_default,
    confirm_callback=confirm_gui_quick,
):
    """Execute a test and compare saved routing to its intended path.

    Parameters
    ----------
    session : RAGSession
        Active session; use a new conversation for independent groups.
    expected_intent : str
        Intended name in ``literature.json``; never forced on the router.
    question : str
        Test request sent unchanged to KíLín.
    choice_callback : callable or None, optional
        Scripted GUI choice. The default shows lists in the chat.
    confirm_callback : callable or None, optional
        Scripted GUI confirmation. Quick is the default to limit work.

    Returns
    -------
    str or None
        Answer, or None when this test failed before an answer was saved.

    """
    definitions = _intent_definitions()
    if expected_intent not in definitions:
        raise ValueError(f"Unknown literature intent: {expected_intent}")
    expected_tool = definitions[expected_intent].get("tool", {}).get("name")
    count_before = len(session.turns)
    record = {
        "expected": expected_intent,
        "expected_tool": expected_tool,
        "question": question,
    }
    try:
        answer = ask_question(
            session,
            question,
            choice_callback=choice_callback,
            confirm_callback=confirm_callback,
        )
        if len(session.turns) != count_before + 1:
            raise RuntimeError("The question did not save exactly one turn.")
        parameters = session.turns[-1].parameters
        tool = parameters.get("tool", {})
        actual_tool = tool.get("name") if isinstance(tool, dict) else None
        record.update(
            actual=parameters.get("intent"),
            actual_tool=actual_tool,
            invalid_citations=parameters.get("invalid_citations", []),
        )
        record["status"] = (
            "MATCH"
            if record["actual"] == expected_intent
            and actual_tool == expected_tool
            else "MISMATCH"
        )
        if isinstance(tool, dict):
            record["output"] = tool.get("saved") or tool.get("report")
            required = _TOOL_RESULT_FIELDS.get(expected_tool, ())
            missing = [key for key in required if key not in tool]
            if record["status"] == "MATCH" and missing:
                record["status"] = "INCOMPLETE"
                record["missing"] = missing
            # Save and report cases should create a file if advertised.
            saved = tool.get("saved")
            if saved and not (project_dir / saved).is_file():
                record["status"] = "INCOMPLETE"
                record["missing_file"] = saved
    except Exception as error:
        answer = None
        record["status"] = "ERROR"
        record["error"] = f"{type(error).__name__}: {error}"
        print(f"INTENT TEST ERROR: {record['error']}")
    intent_checks.append(record)
    print(
        f"INTENT TEST: {expected_intent} -> {record.get('actual', '-')}; "
        f"tool {expected_tool or '-'} -> "
        f"{record.get('actual_tool') or '-'}; {record['status']}"
    )
    if record.get("missing"):
        print(f"  Missing tool result fields: {record['missing']}")
    if record.get("missing_file"):
        print(f"  Missing output file: {record['missing_file']}")
    if record.get("invalid_citations"):
        print(f"  Invalid citations: {record['invalid_citations']}")
    if record.get("output"):
        print(f"  Output file: {record['output']}")
    return answer


def skip_intent(expected_intent, reason):
    """Record a test intentionally not executed in the current run."""
    if expected_intent not in _intent_definitions():
        raise ValueError(f"Unknown literature intent: {expected_intent}")
    intent_checks.append(
        {"expected": expected_intent, "status": "SKIPPED",
         "reason": reason}
    )
    print(f"SKIPPED {expected_intent}: {reason}")


def show_intent_coverage():
    """Report attempted coverage and actual tool/intent agreement.

    Returns
    -------
    dict
        Summary counts: matched, mismatched, errors, skipped, untested.

    """
    definitions = _intent_definitions()
    counts = {key: 0 for key in (
        "matched", "mismatched", "incomplete", "errors",
        "skipped", "untested"
    )}
    print("\nLITERATURE INTENT COVERAGE")
    print(f"{'Intent':<27} {'Expected tool':<24} Status")
    print("-" * 79)
    for intent, settings in definitions.items():
        records = [
            entry for entry in intent_checks
            if entry["expected"] == intent
        ]
        active = [entry for entry in records
                  if entry["status"] != "SKIPPED"]
        if not records:
            status = "NOT RUN"
            counts["untested"] += 1
        elif not active:
            status = "SKIPPED"
            counts["skipped"] += 1
        elif all(entry["status"] == "MATCH" for entry in active):
            status = "MATCH"
            counts["matched"] += 1
        elif any(entry["status"] == "ERROR" for entry in active):
            status = "ERROR"
            counts["errors"] += 1
        elif any(entry["status"] == "INCOMPLETE" for entry in active):
            status = "INCOMPLETE"
            counts["incomplete"] += 1
        else:
            status = "MISMATCH"
            counts["mismatched"] += 1
        expected_tool = settings.get("tool", {}).get("name") or "model"
        print(f"{intent:<27} {expected_tool:<24} {status}")
    print("-" * 79)
    print(f"Routing/tool matches: {counts['matched']}/{len(definitions)}")
    print("Note: matching a route does not validate scientific correctness.")
    return counts



## Create Empty Literature Project

In [5]:
from pathlib import Path

from kilin.workspace import prepare_sources_workspace

# Locate the repository independently of the notebook working directory.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError(
            "Run this notebook from within the KíLín repository."
        )
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / "trial_literature"
project_dir.parent.mkdir(parents=True, exist_ok=True)

# Do not overwrite a project created by an earlier run.
if project_dir.exists():
    raise FileExistsError(f"Project directory already exists: {project_dir}")

project = prepare_sources_workspace(
    [],
    "literature",
    workspace_directory=project_dir,
    settings={"allow_download": False},
)

print(f"Created literature project: {project.root}")
print(f"Source directory: {project.source}")

Created literature project: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature
Source directory: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/sources


## First Conversation without Sources

In [6]:
store, conversation, session = create_conversation(project_dir)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T22-35-09.json
Answering model: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
Router model: Qwen/Qwen3-4B-MLX-4bit


In [7]:
answer = ask_question(session, "hello")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
hello

ASSISTANT:
Hi there! How can I help you today?

Resolved intent: small_talk
Wall time: 6.823 s

TURN: turn_0001

EXCHANGE
  question:
hello


ANSWER
  summary:
Hi there!


INTENT
  intent:
small_talk

  standalone:
True

  refers_back:
False

  related_turn_ids:
[]


ROUTING
  shortcut:
(hi|hello|hey|hiya|hej|yo|greetings|good (morning|afternoon|evening|day))(
there| kilin)?

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.001


RETRIEVED CHUNKS
  No passage text found in the retrieval trace.

MODEL CALL: 04_llm_input/01_llm_call_input
  call_id:
llm-call_8cbfef757d6b

  model_id:
lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly

  backend:
mlx

 

In [8]:
answer = ask_question(session, "What can you help me with?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What can you help me with?

ASSISTANT:
I can help you with a wide range of tasks, including answering questions,
writing code, analyzing data, and creative brainstorming.

Resolved intent: small_talk
Wall time: 11.844 s

TURN: turn_0002

EXCHANGE
  question:
What can you help me with?


ANSWER
  summary:
I can help you with a wide range of tasks, including answering questions,
writing code, analyzing data, and creative brainstorming.


INTENT
  intent:
small_talk

  standalone:
True

  refers_back:
False

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.008


RETRIEVED CHUNKS
  No passage text foun

In [9]:
answer = ask_question(session, "What is Brownian motion?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
What is Brownian motion?

ASSISTANT:
The user is asking "What is Brownian motion?" This is a general knowledge
question about a physics/mathematics concept. The system prompt says to "Answer
only from the supplied project evidence." However, there is no project evidence
supplied in this conversation. The user is asking a general knowledge question.

Wait, let me re-read the system prompt: "Answer only from the supplied project
evidence. Put source numbers such as [1] and [2] directly after supported
claims. Do not write a separate reference list or copy bibliographic text. Say
when evidence is insufficient."

There is no project evidence provided in this conversation. The user is asking
a general knowledge question about Brownian motion. Since there's no supplied
project evidence, I should say the evidence is insufficient.

But wait - this is a general knowledge question. The instruction says "Answer
only from the supplied project evidence." Since no project evidence has been
su

In [10]:
answer = ask_question(session, "Which articles are available in this project?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Which articles are available in this project?

ASSISTANT:
This project has **0 source files**, all with searchable text.

**By type**


**0 files**


Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Resolved intent: project_inventory
Executed tool: project_inventory
Wall time: 4.787 s

TURN: turn_0004

EXCHANGE
  question:
Which articles are available in this project?


ANSWER
  summary:
This project has **0 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

 

In [11]:
answer = ask_question(session, "Summarize the articles in this project.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
Summarize the articles in this project.

ASSISTANT:
This project has **0 source files**, all with searchable text.

**By type**


Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Resolved intent: project_inventory
Executed tool: project_inventory
Wall time: 4.855 s

TURN: turn_0005

EXCHANGE
  question:
Summarize the articles in this project.


ANSWER
  summary:
This project has **0 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  dura

In [12]:
answer = ask_question(
    session,
    "Find articles about neural networks in this project.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Find articles about neural networks in this project.

ASSISTANT:
Found **0 sentences** about “neural networks” in **0 of 0 files**.
No sentence in scope is about this topic.

Resolved intent: topic_listing
Executed tool: find_passages
Wall time: 6.925 s

TURN: turn_0006

EXCHANGE
  question:
Find articles about neural networks in this project.


ANSWER
  summary:
Found **0 sentences** about “neural networks” in **0 of 0 files**.


INTENT
  intent:
topic_listing

  standalone:
False

  refers_back:
True

  related_turn_ids:
[5]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.002


RETRIEVED CHUNKS
  No passage text found

In [13]:
session.close()

## Add Articles to Project

In [14]:
from kilin.workspace import add_project_sources

fixtures = repo_root / "tests" / "fixtures"

articles = sorted(fixtures.glob("*.pdf"))

if not articles:
    raise FileNotFoundError(f"No PDF articles found in {fixtures}")

project = add_project_sources(
    project_dir,
    articles,
)

print(f"Project: {project.root}")
print(f"Imported articles: {len(articles)}")

for article in sorted(project.source.glob("*.pdf")):
    print(f"  {article.name}")

Project: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature
Imported articles: 5
  mijalkov17braph.pdf
  mijalkov25computational.pdf
  patil25deep.pdf
  tiunn26braph2.pdf
  yao25connecting.pdf


## Document Discovery and Retrieval

In [15]:
store, conversation, session = create_conversation(project_dir)

Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T22-36-33.json
Answering model: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
Router model: Qwen/Qwen3-4B-MLX-4bit


### GUI-equivalent choices for long mention lists

The next two calls explicitly select the GUI's **Show first 20** and
**Save all** options. A third call disables the callback and exercises
the notebook-only `MAX_LISTED` fallback (100). The selected output limit
does not change the total number of matching mentions.


In [16]:
answer = ask_question(session, "Which articles are available in this project?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Which articles are available in this project?

ASSISTANT:
This project has **5 source files**, all with searchable text.

**By type**

- PDF: 5

**5 files**

- mijalkov17braph.pdf (78 parts, 15.8 MB)
- mijalkov25computational.pdf (93 parts, 6.3 MB)
- patil25deep.pdf (114 parts, 6.4 MB)
- tiunn26braph2.pdf (36 parts, 3.1 MB)
- yao25connecting.pdf (130 parts, 3.5 MB)

Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Resolved intent: project_inventory
Executed tool: project_inventory
Wall time: 7.968 s

TURN: turn_0001

EXCHANGE
  question:
Which articles are available in this project?


ANSWER
  summary:
This project has **5 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTIN

In [17]:
# GUI choice: display the first 20 mentions.
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=choose_first_20,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)



USER:
Which of these articles discuss BRAPH?

GUI choice requested: Found 183 mentions of “BRAPH”. Save the full list as a Markdown file?
    0: Save all 183 to outputs/ as Markdown
  * 1: Show the first 20 here
  Decision: option 1
ASSISTANT:
Found **183 mentions** of “BRAPH” in **5 of 5 files** (ignoring case, whole
words).

**Mentions per file**

- tiunn26braph2.pdf: 85
- mijalkov17braph.pdf: 80
- mijalkov25computational.pdf: 9
- patil25deep.pdf: 7
- yao25connecting.pdf: 2

**Where**

1. mijalkov17braph.pdf, page 1, line 3: **BRAPH**: A graph theory software for
the analysis of brain connecti… [1]
2. mijalkov17braph.pdf, page 1, line 27: …, for the Alzheimer’s Disease
Neuroimaging Initiative (2017) **BRAPH**: A graph theory software for the
analysis of brain [2]
3. mijalkov17braph.pdf, page 1, line 66: …ecture. We have developed a freeware
MatLab-based software (**BRAPH**–BRain Analysis using graPH the- ory) for
connectivity analy… [3]
4. mijalkov17braph.pdf, page 1, line 69: …sion

In [18]:
# GUI choice: save the complete list as markdown.
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=choose_save_all,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)



USER:
Which of these articles discuss BRAPH?

GUI choice requested: Found 183 mentions of “BRAPH”. Save the full list as a Markdown file?
  * 0: Save all 183 to outputs/ as Markdown
    1: Show the first 20 here
  Decision: option 0
ASSISTANT:
Found **183 mentions** of “BRAPH” in **5 of 5 files** (ignoring case, whole
words).

**Mentions per file**

- tiunn26braph2.pdf: 85
- mijalkov17braph.pdf: 80
- mijalkov25computational.pdf: 9
- patil25deep.pdf: 7
- yao25connecting.pdf: 2

Scanned all indexed text in 5 files.

The full list of 183 mentions is saved as `outputs/mentions_braph.md`.

Resolved intent: mention_listing
Executed tool: scan_and_list
Wall time: 6.367 s

TURN: turn_0003

EXCHANGE
  question:
Which of these articles discuss BRAPH?


ANSWER
  summary:
Found **183 mentions** of “BRAPH” in **5 of 5 files** (ignoring case, whole
words).


INTENT
  intent:
mention_listing

  standalone:
False

  refers_back:
True

  related_turn_ids:
[2]


ROUTING
  shortcut:
None

  fallback:
No

In [19]:
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=None,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)



USER:
Which of these articles discuss BRAPH?

ASSISTANT:
We need answer user's question: "Which of these articles discuss BRAPH?"
Evidence provided [1]-[10]. Need answer only from supplied project evidence.
Put source numbers directly after supported claims. No separate reference list.
Say when evidence insufficient. Need likely list articles among provided
evidence that discuss BRAPH. Need identify which articles discuss BRAPH.
Evidence includes:
[1] mijalkov17braph.pdf:1-15 context related: title BRAPH: A graph theory
software... discusses BRAPH.
[2] mijalkov17braph.pdf:31-48: presents BRAPH.
[3] tiunn26braph2.pdf:64-81: BRAPH 2.
[4] mijalkov17braph.pdf:22-44: Overview of BRAPH.
[5] mijalkov17braph.pdf:58-71: BRAPH.
[6] tiunn26braph2.pdf:18-36: BRAPH 2.
[7] mijalkov25computational.pdf:1-26: no BRAPH in excerpt? It discusses
reservoir computing, no BRAPH. But earlier system said 9 mentions in
mijalkov25computational.pdf. However evidence [7] doesn't mention BRAPH. Need
answer from ev

In [20]:
answer = ask_question(
    session,
    "Show me the source information for the BRAPH 2 article.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
Show me the source information for the BRAPH 2 article.

ASSISTANT:
<think>Model call 1:
We need answer user's request. Need use only source text shown, cite like [1],
say plainly when text doesn't make clear. User: "Show me the source information
for the BRAPH 2 article." Source text includes [1] mijalkov17braph.pdf, [2]
same, [3] same. They ask source information for the BRAPH 2 article. Need
likely identify source info: filename mijalkov17braph.pdf, PLOS ONE |
https://doi.org/10.1371/journal.pone.0178798 August 1, 2017 from [3]. But
"BRAPH 2 article"? Maybe they mean the article about BRAPH? Source text says
PLOS ONE ... August 1, 2017. Need show source information. We can say: The
source information shown is: mijalkov17braph.pdf [1][2][3], and the article
appears to be published in PLOS ONE, DOI
https://doi.org/10.1371/journal.pone.0178798, dated August 1, 2017 [3]. The
text does not explicitly say "BRAPH 2" or give full bibliographic details
(authors, title, volume, pages).

In [21]:
answer = ask_question(
    session,
    "Find passages discussing graph measures in the BRAPH articles.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Find passages discussing graph measures in the BRAPH articles.

ASSISTANT:
Found **0 mentions** of “graph measures” in **tiunn26braph2.pdf** (ignoring
case, whole words).

Scanned only tiunn26braph2.pdf, the best match for “BRAPH articles”. Name the
file to scan a different one.

Scanned all indexed text in tiunn26braph2.pdf.

Resolved intent: mention_listing
Executed tool: scan_and_list
Wall time: 7.227 s

TURN: turn_0006

EXCHANGE
  question:
Find passages discussing graph measures in the BRAPH articles.


ANSWER
  summary:
Found **0 mentions** of “graph measures” in **tiunn26braph2.pdf** (ignoring
case, whole words).


INTENT
  intent:
mention_listing

  standalone:
False

  refers_back:
False

  related_turn_ids:
[4]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

In [22]:
answer = ask_question(
    session,
    "What do those passages say about their implementation?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)


USER:
What do those passages say about their implementation?

ASSISTANT:
<think>Model call 1:
We need answer user's request. Need explain cited sentence using surrounding
sentences, short. User: "Source text ... [3] ... >>From the clinical point of
view, BRAPH presents the following strengths: ...<< From the user point of
view, BRAPH is the only fully vertically integrated software that allows
carrying out all the steps of a graph theory analysis, from importing the
neuroimaging data to saving the final results and the analysis parameters in a
PLOS ONE | https://doi.org/10.1371/journal.pone.0178798 August 1, 2017

Question: What do those passages say about their implementation?"

Need use only text shown, cite like [1], say plainly when text does not make
something clear. Need explain what cited sentence is talking about, using
sentences around it. Question asks what do those passages say about their
implementation? The cited sentence is clinical strengths. Surrounding
sentences: [3] 

In [23]:
answer = ask_question(session, "Which articles discuss deep learning?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)


USER:
Which articles discuss deep learning?

GUI plan: find_passages | Read all 3710 sentences in 5 files and list the ones about “deep learning”.
  Items: 3710, batches: 44, quick items: 24
  Decision: continue
ASSISTANT:
**4 of 5 files** match “deep learning”, from 92 matching sentences.
- **patil25deep.pdf** (43 sentences): “Deep-Learning Investigation of
Vibrational Raman Spec- tra for Plant-Stress Analysis” [21]
- **mijalkov25computational.pdf** (32 sentences): “Here we use reservoir
computing10—a machine-learning framework that can reproduce the functional
dynamics of biological networks11–14—to characterize how the cognitive function
of the human brain deterio- rates with aging.” [1]
- **tiunn26braph2.pdf** (12 sentences): “Keywords: Network analyses
Neuroscience Deep learning Specialized distributions Custom pipelines
Reproducibility” [31]
- **yao25connecting.pdf** (5 sentences): “Using fMRI as an orthogonal modality,
we show that a data-driven model trained to classify schizo

In [24]:
session.close()

## Article Understanding and Synthesis

In [25]:
store, conversation, session = create_conversation(project_dir)

Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T23-12-54.json
Answering model: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
Router model: Qwen/Qwen3-4B-MLX-4bit


In [26]:
answer = ask_question(session, "Give me an overview of the BRAPH 2 paper.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Give me an overview of the BRAPH 2 paper.

GUI choice requested: Which paper should I summarize?
  * 0: tiunn26braph2.pdf (best match for the question)
    1: mijalkov17braph.pdf (matches the question)
    2: mijalkov25computational.pdf (in this project)
    3: patil25deep.pdf (in this project)
  Decision: option 0
ASSISTANT:
We need answer user's request: "Give me an overview of the BRAPH 2 paper." Need
use only passages, cite like [3], never add results they do not state. Need
likely 150-300 words, concise, a few sentences on what paper does, how, what it
finds. Need cite passages. Need ensure no unsupported. Let's extract key
points.

Paper: BRAPH 2: flexible, open-source, reproducible, community-oriented,
easy-to-use framework for network analyses in neurosciences [1]. It addresses
growing complexity/size of network analyses needing unbiased reproducible
insights [2]. It is versatile open-source framework offering streamlined
workflows for advanced statistical models and dee

In [27]:
answer = ask_question(
    session,
    "Summarize its main contributions and limitations.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
Summarize its main contributions and limitations.

ASSISTANT:
We need answer user's request: summarize main contributions and limitations of
paper. Need use only passages, cite like [3], etc. Need likely 150-300 words.
Need concise but detailed enough. Need include main contributions and
limitations. Need cite passages. Let's extract.

Paper: BRAPH 2: flexible open-source reproducible community-oriented
easy-to-use framework for network analyses in neurosciences. Contributions:
open-source framework, streamlined workflows for advanced statistical models
and deep learning, community-oriented extension/reuse/sharing [2]. Genesis
compiler builds specialized distributions with custom pipelines [2]. Adds
multiplex graph analysis, deep learning pipelines, Genesis-based generation of
reusable pipelines and specialized distributions compared to BRAPH 1 [6].
Genesis compiles structured element descriptions into executable MATLAB
classes, GUIs, unit tests, supports integrated statistical 

In [28]:
answer = ask_question(
    session,
    "Generate a detailed summary of the BRAPH 2 paper.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
Generate a detailed summary of the BRAPH 2 paper.

ASSISTANT:
Done: the technical summary of tiunn26braph2.pdf is saved as
`outputs/tiunn26braph2_summary.md`.

---

# BRAPH 2: A Genesis-Driven Framework for Reproducible Network Neuroscience and
Custom Software Generation

**Source paper:** Tiunn, I.-U. et al. BRAPH 2: A flexible, open-source,
reproducible, community-oriented, easy-to-use framework for network analyses in
neurosciences. (2026).

## Technical Problem and Contribution
Network analyses in neuroscience face fragmentation due to tools focusing on
specific tasks, which complicates workflow integration and delays the adoption
of new robust, reproducible features. Existing software often requires manual
intervention to build full workflows, a limitation exacerbated by rapid
advances in AI. BRAPH 2 addresses this by providing a flexible, open-source
framework that combines streamlined workflows for advanced statistical models
and deep learning with a community-oriented ex

In [29]:
answer = ask_question(
    session,
    "How does BRAPH 2 differ from the earlier BRAPH framework?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
How does BRAPH 2 differ from the earlier BRAPH framework?

ASSISTANT:
We need answer user's question: "How does BRAPH 2 differ from the earlier BRAPH
framework?" Need use supplied evidence only. Need compare point by point, say
where agree/differ? The system says compare sources point by point, and say
where they agree and where they differ. But user asks specific. Need answer
concise 150-300 words maybe. Need cite source numbers immediately after claims.
Need not separate reference list. Need only from evidence. Need likely mention
BRAPH 1 vs BRAPH 2. Evidence [8] explicitly: Compared with BRAPH 1, it adds
multiplex graph analysis, deep learning pipelines, and Genesis-based generation
of reusable pipelines and specialized distributions. Unlike tools focused
mainly on individual analysis steps, BRAPH 2 centers on Genesis module,
compiles structured element descriptions into executable MATLAB classes, GUIs,
unit tests, supporting integrated statistical testing and reproducible
wo

In [30]:
answer = ask_question(session, "What evidence supports those differences?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
What evidence supports those differences?

GUI plan: reference_check | Check 12 claims of tiunn26braph2.pdf against the 3 cited papers in the project, one model call each.
  Items: 12, batches: 12, quick items: 5
  Decision: continue
ASSISTANT:
<think>Model call 1:
We need answer user's request. Need judge whether cited work [19] supports
sentence part: "BRAPH 1 [19]" in list of free and open-source software
platforms for brain connectivity analysis. Need choose verdict among supported,
partly supported, not supported, contradicted, background, not found. Need
output exactly three lines: VERDICT, QUOTE, COMMENT.

We need analyze. Citing sentence: "Many free and open-source software platforms
for brain connectivity analysis exist, such as the Brain Connectivity Toolbox
[13], eConnectome [14], GAT [15], CONN [16], BrainNet Viewer [17], GRETNA [18],
and BRAPH 1 [19]." Reference [19] cited for "BRAPH 1". Context: "A reference
table comparing BRAPH 2 with these related software platf

In [31]:
answer = ask_question(
    session,
    "Which of the four papers is most relevant to graph-based analysis "
    "of brain connectivity, and why?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Which of the four papers is most relevant to graph-based analysis of brain
connectivity, and why?

ASSISTANT:
The user asks: "Which of the four papers is most relevant to graph-based
analysis of brain connectivity, and why?"

Let me identify the four papers from the evidence:
1. mijalkov17braph.pdf - BRAPH: A graph theory software for the analysis of
brain connectivity (2017)
2. tiunn26braph2.pdf - BRAPH 2 paper (2026)
3. mijalkov25computational.pdf - Computational memory capacity predicts aging
and cognitive decline (2025)
4. yao25connecting.pdf - Connecting genomic results for psychiatric disorders
to human brain cell types and regions (2025)

Now, which is most relevant to "graph-based analysis of brain connectivity"?

Let me look at the evidence:

- mijalkov17braph.pdf [1][2][3][5][6][8][9][10][16][19]: This is explicitly
titled "BRAPH: A graph theory software for the analysis of brain connectivity."
It describes a complete software package for graph theoretical analysis,
in

In [32]:
session.close()

## Specialized Literature Tools

In [33]:
store, conversation, session = create_conversation(project_dir)

Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-02T02-28-07.json
Answering model: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
Router model: Qwen/Qwen3-4B-MLX-4bit


In [34]:
answer = ask_question(
    session,
    "Find the passage in the BRAPH 2 paper "
    "describing its software architecture.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Find the passage in the BRAPH 2 paper describing its software architecture.

ASSISTANT:
I couldn't find an earlier answer with sources to show. Ask the question first,
then ask to see its source.

Resolved intent: source_context
Executed tool: show_source
Wall time: 7.298 s

TURN: turn_0001

EXCHANGE
  question:
Find the passage in the BRAPH 2 paper describing its software architecture.


ANSWER
  summary:
I couldn't find an earlier answer with sources to show.


INTENT
  intent:
source_context

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
relevant

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.001


RETR

In [35]:
answer = ask_question(
    session,
    "What programming language is BRAPH 2 implemented in?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What programming language is BRAPH 2 implemented in?

ASSISTANT:
The user asks: "What programming language is BRAPH 2 implemented in?"

Looking at the evidence:

[1] states: "Because BRAPH 2 is implemented in MATLAB, large voxel-level
networks may be limited by memory demands..."

[8] states: "Because BRAPH 2 is implemented in" (cut off, but clearly leading
to MATLAB based on context)

[9] states: "Written in MATLAB, BRAPH 2 is portable across operating systems
and supports multiple data types beyond brain imaging."

[3] states: "These deep learning pipelines are implemented in MATLAB within
BRAPH 2 and rely on the MATLAB Deep Learning Toolbox."

[4] mentions "MATLAB versions (R2022a, R2022b, R2023a, R2023b, R2024a, and
R2024b)"

So the answer is clearly MATLAB. Multiple sources confirm this.

The question is simple and direct. I'll answer concisely.
</think>

BRAPH 2 is implemented in MATLAB [1][9]. The paper explicitly states, "Written
in MATLAB, BRAPH 2 is portable across ope

In [36]:
answer = ask_question(session, "List the articles that mention graph theory.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
List the articles that mention graph theory.

GUI choice requested: Found 65 mentions of “graph theory”. Save the full list as a Markdown file?
    0: Save all 65 to outputs/ as Markdown
  * 1: Show the first 20 here
  Decision: option 1
ASSISTANT:
Found **65 mentions** of “graph theory” in **5 of 5 files** (ignoring case,
whole words).

**Mentions per file**

- mijalkov17braph.pdf: 59
- tiunn26braph2.pdf: 3
- mijalkov25computational.pdf: 1
- patil25deep.pdf: 1
- yao25connecting.pdf: 1

**Where**

1. mijalkov17braph.pdf, page 1, line 3: BRAPH: A **graph theory** software for
the analysis of brain connectivity [1]
2. mijalkov17braph.pdf, page 1, lines 27-28: … Alzheimer’s Disease Neuroimaging
Initiative (2017) BRAPH: A **graph theory** software for the analysis of brain
connectivity. PLoS ONE 12… [2]
3. mijalkov17braph.pdf, page 1, line 63: …ain network has been studied
extensively using concepts from **graph theory**, where the brain is
represented as a set of nodes connected… [

In [37]:
answer = ask_question(
    session,
    "Filter those results to articles discussing brain connectivity.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Filter those results to articles discussing brain connectivity.

GUI plan: find_passages | Read all 3710 sentences in 5 files and list the ones about “brain connectivity”.
  Items: 3710, batches: 44, quick items: 20
  Decision: continue
ASSISTANT:
**5 of 5 files** match “brain connectivity”, from 261 matching sentences.
- **mijalkov17braph.pdf** (109 sentences): “BRAPH: A graph theory software for
the analysis of brain connectivity” [1]
- **mijalkov25computational.pdf** (80 sentences): “Therefore, these ﬁndings
show that the age-related reductions in memory capacity are most evident when
an increasing number of weak connections are present in the network.” [51]
- **yao25connecting.pdf** (47 sentences): “Connecting genomic results for
psychiatric disorders to human brain cell types and regions reveals convergence
with functional connectivity” [104]
- **tiunn26braph2.pdf** (24 sentences): “BRAPH 2: A flexible, open-source,
reproducible, community-oriented, easy-to-use framework fo

In [38]:
answer = ask_question(
    session,
    "Classify the five articles by their main research topic.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
Classify the five articles by their main research topic.

GUI plan: find_passages | Read all 3710 sentences in 5 files and list the ones about “main research topic”.
  Items: 3710, batches: 44, quick items: 309
  Decision: continue
ASSISTANT:
**5 of 5 files** match “main research topic”, from 448 matching sentences.
- **yao25connecting.pdf** (146 sentences): “Connecting genomic results for
psychiatric disorders to human brain cell types and regions reveals convergence
with functional connectivity” [88]
- **mijalkov25computational.pdf** (127 sentences): “Computational memory
capacity predicts aging and cognitive decline” [21]
- **patil25deep.pdf** (92 sentences): “To better understand and manage plant
stress, advanced sensing technologies have been pro-” [57]
- **mijalkov17braph.pdf** (57 sentences): “BRAPH: A graph theory software for
the analysis of brain connectivity” [1]
- **tiunn26braph2.pdf** (26 sentences): “BRAPH 2 is built on Genesis and is
primarily designed for region-

In [39]:
answer = ask_question(
    session,
    "Review the five articles for their relevance to a literature review "
    "on computational brain connectivity.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Review the five articles for their relevance to a literature review on
computational brain connectivity.

ASSISTANT:
The user asks me to review five articles for their relevance to a literature
review on computational brain connectivity. I need to use only the provided
evidence [1]-[15] and classify each of the five articles by relevance.

The five articles are:
1. mijalkov17braph.pdf - BRAPH: A graph theory software for the analysis of
brain connectivity
2. mijalkov25computational.pdf - Computational memory capacity predicts aging
and cognitive decline
3. yao25connecting.pdf - Connecting genomic results for psychiatric disorders
to human brain cell types and regions reveals convergence with functional
connectivity
4. tiunn26braph2.pdf - BRAPH 2: A flexible, open-source, reproducible,
community-oriented, easy-to-use framework for network analyses in neurosciences
5. patil25deep.pdf - This appears to be about precision agriculture / plant
stress detection using Raman spectroscopy

In [40]:
answer = ask_question(
    session,
    "Check whether the BRAPH 2 paper supports the claim "
    "that it provides a graphical user interface.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)


USER:
Check whether the BRAPH 2 paper supports the claim that it provides a graphical
user interface.

GUI choice requested: Which paper's references should I check?
  * 0: mijalkov17braph.pdf (used in turn 6)
    1: mijalkov25computational.pdf (used in turn 6)
    2: tiunn26braph2.pdf (best match for the question)
    3: patil25deep.pdf (in this project)
  Decision: option 0
ASSISTANT:
**Reference check of mijalkov17braph.pdf**

57 references (numbered as printed). In the project: 0 found, 0 probable, 56
missing, 1 too short to identify. 70 bracketed citations in the text cite -814
of them.

**Missing from the project:**

- **1.** Sporns O. The human connectome: origins and challenges. Neuroimage
2013; 80: 53–61. https://doi. org/10.1016/j.neuroimage.2013.03.023 PMID:
23528922 — [DOI (as printed;
unverified)](https://doi.org/10.1016/j.neuroimage.2013.03.023)
- **2.** Watts DJ, Strogatz SH. Collective dynamics of ’small-world’ networks.
Nature 1998; 393: 440–442. https://doi.org/10.10

In [41]:
answer = ask_question(
    session,
    "Which article should I cite for the introduction of the BRAPH framework?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)


USER:
Which article should I cite for the introduction of the BRAPH framework?

ASSISTANT:
<think>Model call 1:
We need answer user's request. Need judge if candidate work can be cited for
sentence: "Which article should I cite for the introduction of the BRAPH
framework?" Passages from tiunn26braph2.pdf. Need choose verdict among
supported, partly supported, not supported, contradicted, background, not
found. Need output exactly three lines: VERDICT, QUOTE, COMMENT.

We need analyze. Sentence asks which article should I cite for introduction of
BRAPH framework? It is a question, not a claim? The writer needs reference for
sentence? Actually sentence: "Which article should I cite for the introduction
of the BRAPH framework?" They need a reference for introduction of BRAPH
framework. Candidate work is about BRAPH 2, an open-source framework, Genesis
module, etc. Does it state that it is the article to cite for introduction of
BRAPH framework? The passages describe BRAPH 2 as framework,

In [42]:
session.close()

## Continuity and Robustness

In [43]:
store, conversation, session = create_conversation(project_dir)

Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-02T04-27-02.json
Answering model: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
Router model: Qwen/Qwen3-4B-MLX-4bit


In [44]:
answer = ask_question(session, "Summarize the BRAPH 2 article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Summarize the BRAPH 2 article.

GUI choice requested: Which paper should I summarize?
  * 0: tiunn26braph2.pdf (best match for the question)
    1: mijalkov17braph.pdf (matches the question)
    2: mijalkov25computational.pdf (in this project)
    3: patil25deep.pdf (in this project)
  Decision: option 0
ASSISTANT:
We need answer user's request: summarize BRAPH 2 article. Need use only
passages, cite like [3], never add results they do not state. Need likely
150-300 words, concise, a few sentences on what paper does, how, and what it
finds. Need cite passages. Need ensure no unsupported claims. Let's extract key
points from passages.

Paper: SoftwareX 35 (2026) 102791, BRAPH 2: flexible, open-source,
reproducible, community-oriented, easy-to-use framework for network analyses in
neurosciences [1]. Abstract: network analyses grow complexity/size, need
unbiased reproducible insights. BRAPH 2 versatile open-source framework
offering streamlined workflows for advanced statistical mo

In [45]:
answer = ask_question(session, "What are its main limitations?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What are its main limitations?

ASSISTANT:
We need answer user's question: "What are its main limitations?" Context:
previous asked summarize BRAPH 2 article. Now asks "its main limitations"
likely refers to BRAPH 2 article. Need use supplied evidence. Need prioritize
direct evidence about requested target. Evidence includes [2]
tiunn26braph2.pdf:28-49 related, [9] tiunn26braph2.pdf:19-39 related. Also [5]
and [8] are mijalkov25computational.pdf limitations about memory capacity
study, not BRAPH 2? The question says "its" after summarizing BRAPH 2 article,
so likely BRAPH 2's main limitations. Need answer from evidence. Direct
evidence: [9] says because BRAPH 2 implemented in MATLAB, large voxel-level
networks may be limited by memory demands, although sparse-matrix
representations can help mitigate at cost of some computational efficiency.
Further limitation: Genesis, software-generating core, not naturally suited to
direct benchmarking against conventional analysis tools; benc

In [46]:
answer = ask_question(session, "How does it compare with the 2017 paper?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
How does it compare with the 2017 paper?

ASSISTANT:
We need answer user's question: "How does it compare with the 2017 paper?" They
provided evidence snippets. Need use only supplied project evidence. Need put
source numbers directly after supported claims. No separate reference list.
Compare sources point by point, say where agree and differ. Need likely compare
BRAPH 2 article with 2017 BRAPH paper (mijalkov17braph.pdf). Need summarize
comparison. Need maybe mention 2017 paper describes BRAPH (v1) as complete
software package, GUI, object-oriented modular, strengths: permutation testing,
visualization, longitudinal, modular, multimodal, fully vertically integrated
from import to saving results. BRAPH 2 article says compared with BRAPH 1 adds
multiplex graph analysis, deep learning pipelines, Genesis-based generation of
reusable pipelines and specialized distributions; addresses reproducibility,
maintainability, long-term reliability; combines user-friendly environment with
so

In [47]:
answer = ask_question(session, "Now tell me about the deep learning article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Now tell me about the deep learning article.

ASSISTANT:
We need answer user's question: "Now tell me about the deep learning article."
Need use supplied evidence. Need likely identify which article is "the deep
learning article"? Evidence includes [2] patil25deep.pdf:1-12, [5]
patil25deep.pdf:1-26, [8] patil25deep.pdf:22-62. Also [1] tiunn26braph2.pdf
mentions custom deep learning pipeline, recursive feature elimination,
multilayer perceptron and iterative feature elimination, identifies consistent
important brain regions across two datasets. [3] mijalkov25computational.pdf
mentions age prediction using multilayer perceptron, memory capacity. [4]
tiunn26braph2.pdf says BRAPH 2 standard distribution includes deep learning
pipelines for supervised tasks classification, prediction, diagnosis,
implemented in MATLAB Deep Learning Toolbox. [9] says BRAPH 2 adds deep
learning pipelines, used to implement recursive feature elimination pipeline
for deep learning [22] and reservoir compu

In [48]:
answer = ask_question(session, "What methods does it use?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
What methods does it use?

ASSISTANT:
The user asks "What methods does it use?" referring to "the deep learning
article" from the previous turn. In the previous turn, I identified the deep
learning article as patil25deep.pdf. Now I need to look at the evidence
provided to determine what methods that article uses.

Let me look at the evidence:

[4] patil25deep.pdf:28-48 - This describes DIVA, which:
- Does not require manually removing fluorescence background
- Operates in two key steps
- First step: reduces influence of fluorescence by calculating the first
derivative D(ṽ) of raw Raman spectra I(ṽ)
- Avoids need for background correction and normalization
- The derivative of slowly varying fluorescence background is smaller than
derivative of fast varying Raman peaks
- No ad hoc background fitting procedures needed

[6] patil25deep.pdf:28-50 - This describes:
- Clustering (each cluster, reproduction of average characteristic spectra
D(ṽ))
- Reconstruction captures key biomole

In [49]:
answer = ask_question(session, "Compare it with the previous article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Compare it with the previous article.

ASSISTANT:
The user asks to compare "it" (the deep learning article, patil25deep.pdf) with
"the previous article." Looking at the conversation history, the "previous
article" refers to the BRAPH 2 paper (tiunn26braph2.pdf), which was discussed
in the first turn as the "previous article" context. Actually, let me re-read
the conversation.

In the first user message, they asked about "the deep learning article." I
answered about patil25deep.pdf.
In the second user message, they asked "What methods does it use?" - referring
to the deep learning article (patil25deep.pdf).
Now they ask "Compare it with the previous article."

The "previous article" - what is the previous article? Looking at the evidence
provided, the sources include:
- [1] tiunn26braph2.pdf (BRAPH 2)
- [2] mijalkov25computational.pdf
- [3] tiunn26braph2.pdf
- [4] patil25deep.pdf
- [5] yao25connecting.pdf
- [6] mijalkov25computational.pdf
- [7] mijalkov25computational.pdf
- [8] p

In [50]:
answer = ask_question(
    session,
    "Find evidence that all five articles use the same dataset.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)


USER:
Find evidence that all five articles use the same dataset.

ASSISTANT:
The user asks me to find evidence that all five articles use the same dataset.
Let me look at the five articles and their datasets:

1. tiunn26braph2.pdf (BRAPH 2) - This is a software paper. It mentions "two
datasets" in the context of a custom deep learning pipeline identifying brain
regions [1]. It mentions "lifespan imaging data" for the Memory Capacity
distribution [1]. It's a software framework paper, not a dataset-specific
paper.

2. mijalkov25computational.pdf - Uses "Cambridge Centre for Ageing and
Neuroscience (Cam-CAN) cohort" and "Leipzig Mind-Brain-Body Dataset—LEMON" [2].

3. yao25connecting.pdf - This is about schizophrenia GWAS/genomic data. It
mentions "Data set 2 (54 cases, 54 ctrls)" and "Data set 1 (46 cases, 46
ctrls)" [6]. It's about genomic results for brain disorders.

4. mijalkov17braph.pdf - This is about BRAPH (the original). It discusses
thresholding of structural or functional edg

In [51]:
answer = ask_question(session, "Which article did we discuss first?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)


USER:
Which article did we discuss first?

ASSISTANT:
The user asks: "Which article did we discuss first?"

Looking at the conversation history:
1. First user message: "Summarize the BRAPH 2 article." - I summarized BRAPH 2
(tiunn26braph2.pdf).
2. Second user message: "Compare it with the previous article." - I compared
BRAPH 2 with the deep learning article (patil25deep.pdf).
3. Third user message: "Find evidence that all five articles use the same
dataset." - I discussed all five articles.
4. Fourth user message (current): "Which article did we discuss first?"

The first article discussed in this conversation was the BRAPH 2 article
(tiunn26braph2.pdf), as the user's first question was "Summarize the BRAPH 2
article."

The evidence provided is all from tiunn26braph2.pdf, which confirms this is the
BRAPH 2 article.

The answer is straightforward: the BRAPH 2 article (tiunn26braph2.pdf) was
discussed first.
</think>

The BRAPH 2 article (tiunn26braph2.pdf) was the first article we dis

In [52]:
session.close()

## Literature Intent and Tool Coverage

This section verifies **routing and tool selection** for all 21 intents
configured in `kilin/project_types/literature.json`. The test harness
**does not force** an intent: it records the actual intent and tool saved
by `RAGSession.ask()` and prints `MATCH`, `MISMATCH`, or `ERROR`.

The GUI's **choice** and **confirmation** adapters are simulated using
explicit callbacks. The default choice shows up to 20 results in chat;
if asked to choose an ambiguous paper it selects the first offered
paper and prints that decision. The default confirmation for these
coverage tests is **Quick**, to keep batches manageable. Network lookups
are **always declined**, even if Continue is selected for model plans.

Run the groups in order so their contextual follow-ups have evidence.
The extended group (whole-paper scans, classification, reference checks,
and the four output templates) is opt-in because it can take many model
calls and write files. Set `RUN_EXTENDED_CASES = True` to exercise those
nine remaining intents. Inspect each mismatch rather than assuming a
successful answer means the intended operation ran.

A `MATCH` only validates intent and tool selection, **not** the truth
of a generated claim, citation support, report quality, or GUI rendering.
The notebook does not provide the GUI's optional Qt PDF writer; report
tools will still create their Markdown output.


In [53]:
# False: run the initial 12 intent cases; report the rest as SKIPPED.
# True: exercise every configured literature intent and its selected tool.
RUN_EXTENDED_CASES = True

# Static coverage contract: fail if the project's declared intents change.
PLANNED_INTENTS = {
    "small_talk", "question_answering", "compare_sources",
    "report_generation", "conversation_followup", "rephrase_previous",
    "project_inventory", "mention_listing", "refine_results",
    "source_context", "paper_overview", "suggest_reference",
    "topic_listing", "fact_finding", "document_reading",
    "mention_classification", "reference_check", "research_briefing",
    "popular_science_summary", "technical_summary",
    "experimental_protocol",
}
assert PLANNED_INTENTS == set(_intent_definitions()), (
    "Update the test cases when literature.json adds or removes intents."
)
print(f"Planned intent tests: {len(PLANNED_INTENTS)}")


Planned intent tests: 21


### A. Direct answers, comparison, and conversation continuity

These six tests share one conversation so the follow-up and rephrasing
intents have actual preceding answers to use. They test routing, not
whether the answer's scientific content is correct.


In [54]:
store, conversation, session = create_conversation(project_dir)

check_intent(session, "small_talk", "hello")
check_intent(
    session,
    "question_answering",
    "According to the BRAPH 2 paper, why was its software developed?",
)
check_intent(
    session,
    "compare_sources",
    "Compare the methods of mijalkov17braph.pdf and "
    "tiunn26braph2.pdf, including their differences.",
)
check_intent(
    session,
    "report_generation",
    "Write a structured report with headings about methods for "
    "brain connectivity analysis across these papers.",
)
check_intent(
    session,
    "conversation_followup",
    "In your previous report, what was the main limitation you noted?",
)
check_intent(
    session,
    "rephrase_previous",
    "Rewrite your previous answer in two concise bullet points, "
    "without adding any information.",
)

session.close()


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-02T05-16-12.json
Answering model: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
Router model: Qwen/Qwen3-4B-MLX-4bit

USER:
hello

ASSISTANT:
Hi there! How can I help you today?

Resolved intent: small_talk
Wall time: 6.308 s
INTENT TEST: small_talk -> small_talk; tool - -> -; MATCH

USER:
According to the BRAPH 2 paper, why was its software developed?

ASSISTANT:
We need answer user's question: "According to the BRAPH 2 paper, why was its
software developed?" Need use supplied evidence only. Need cite source numbers
directly after supported claims. Need concise 150-300 words maybe. Need
prioritize direct evidence about requested target. Evidence includes [1], [2],
[3], [5], [8], [9] from tiunn26braph2.pdf. Need infer why developed. [9] says:
"reproducibility, maintainability, and long-term reliability [20,21]. As a
result, researchers face difficulties in adopting cutting-ed

### B. File discovery, exact mentions, and prior-result tools

`refine_results` and `source_context` operate on saved evidence from
earlier turns. The 20-item callback matches the GUI's chat-list option.
If retrieval finds no usable mentions, inspect the corresponding tool's
result before interpreting a later failure as a routing defect.


In [55]:
store, conversation, session = create_conversation(project_dir)

check_intent(
    session,
    "project_inventory",
    "Which PDF files are available in this project?",
)
check_intent(
    session,
    "mention_listing",
    "List the sentences in tiunn26braph2.pdf that mention BRAPH.",
    choice_callback=choose_first_20,
)
check_intent(
    session,
    "source_context",
    "Show the exact source text and surrounding context for the "
    "first cited sentence in that list.",
)
check_intent(
    session,
    "refine_results",
    "Of the BRAPH mentions you listed, which ones describe "
    "software features rather than bibliographic citations?",
)

session.close()


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-02T06-23-03.json
Answering model: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
Router model: Qwen/Qwen3-4B-MLX-4bit

USER:
Which PDF files are available in this project?

ASSISTANT:
This project has **5 source files**, all with searchable text.

**By type**

- PDF: 5

**5 PDF files**

- mijalkov17braph.pdf (78 parts, 15.8 MB)
- mijalkov25computational.pdf (93 parts, 6.3 MB)
- patil25deep.pdf (114 parts, 6.4 MB)
- tiunn26braph2.pdf (36 parts, 3.1 MB)
- yao25connecting.pdf (130 parts, 3.5 MB)

Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Resolved intent: project_inventory
Executed tool: project_inventory
Wall time: 7.558 s
INTENT TEST: project_inventory -> project_inventory; tool project_inventory -> project_inventory; MATCH

USER:
List the sentences in tiunn26braph2.pdf that mention BRAPH.

GUI choice requested: 

### C. Paper overview, reference suggestion, and document selection

The third test intentionally omits the filename to exercise the GUI's
paper-selection dialog. The callback chooses `tiunn26braph2.pdf` by
its **actual option label**, not by assuming a fixed index. If the file
is not offered, it cancels instead of picking an unrelated paper.


In [ ]:
store, conversation, session = create_conversation(project_dir)

check_intent(
    session,
    "paper_overview",
    "Give me a short overview of tiunn26braph2.pdf.",
)
check_intent(
    session,
    "suggest_reference",
    "Suggest a reference from this project for the statement: "
    "BRAPH supports graph analysis of brain networks.",
)
# Additional selection-path test; still expects paper_overview.
check_intent(
    session,
    "paper_overview",
    "Give me a short overview of one paper in the collection.",
    choice_callback=choose_document_named("tiunn26braph2.pdf"),
)

session.close()


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-02T06-35-51.json
Answering model: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
Router model: Qwen/Qwen3-4B-MLX-4bit

USER:
Give me a short overview of tiunn26braph2.pdf.

ASSISTANT:
We need answer user's request: "Give me a short overview of tiunn26braph2.pdf."
Need use only passages, cite like [3], never add results they do not state.
Need likely concise 150-300 words? User says short overview. System says if
asks length/format follow; otherwise keep short: a few sentences on what paper
does, how, and what it finds. But also says otherwise keep answer concise:
about 150-300 words, or more when question asks for detail. There is slight
conflict: "short overview" maybe few sentences. Need likely produce 150-300?
The prompt: "If it asks for a length or format (for example "in two
sentences"), follow it exactly; otherwise keep it short: a few sentences on
what the paper does, h

### D. Extended tool tests (opt-in)

Enable `RUN_EXTENDED_CASES` in the setup cell to execute the nine
remaining intents. **Quick** is selected for model-intensive plans;
this verifies the quick route, not exhaustive full-processing behavior.
Reference lookup over the network is always denied. These calls may
still be slow, and report templates create Markdown files in `outputs/`.


In [ ]:
extended_cases = (
    (
        "topic_listing",
        "Which papers apply graph-theory methods to analyze brain "
        "connectivity, based on their meaning rather than keywords?",
    ),
    (
        "fact_finding",
        "Give a complete list of the programming languages and software "
        "dependencies named in tiunn26braph2.pdf.",
    ),
    (
        "document_reading",
        "Read the whole tiunn26braph2.pdf paper from start to end "
        "and identify which software components it describes.",
    ),
    (
        "mention_classification",
        "Classify the mentions of BRAPH in tiunn26braph2.pdf into "
        "(1) descriptions of current software and (2) references "
        "to earlier work, then count each group.",
    ),
    (
        "reference_check",
        "Check the references cited by tiunn26braph2.pdf and determine "
        "which cited papers are present in this project.",
    ),
)

if RUN_EXTENDED_CASES:
    store, conversation, session = create_conversation(project_dir)
    try:
        for intent, question in extended_cases:
            check_intent(
                session, intent, question,
                confirm_callback=confirm_gui_quick,
            )
    finally:
        session.close()
else:
    for intent, _ in extended_cases:
        skip_intent(intent, "Set RUN_EXTENDED_CASES = True")


### E. Four structured report tools (opt-in)

Each case targets a distinct intent but calls the same
`generate_paper_summary` tool with a different template. Markdown
reports are saved in the project workspace. Repeating the cells may
create additional output files. The notebook does not generate PDFs.


In [ ]:
report_cases = (
    (
        "research_briefing",
        "Prepare a research briefing for tiunn26braph2.pdf.",
    ),
    (
        "popular_science_summary",
        "Write a popular science summary of tiunn26braph2.pdf "
        "for a general audience.",
    ),
    (
        "technical_summary",
        "Prepare a detailed technical summary of tiunn26braph2.pdf "
        "for specialists.",
    ),
    (
        "experimental_protocol",
        "Extract an experimental protocol based on tiunn26braph2.pdf; "
        "identify missing replication details without inventing them.",
    ),
)

if RUN_EXTENDED_CASES:
    store, conversation, session = create_conversation(project_dir)
    try:
        for intent, question in report_cases:
            check_intent(
                session, intent, question,
                confirm_callback=confirm_gui_quick,
            )
    finally:
        session.close()
else:
    for intent, _ in report_cases:
        skip_intent(intent, "Set RUN_EXTENDED_CASES = True")


### F. Confirmation and output-path regression (optional)

The main test groups choose **Quick** for large model plans. To test
GUI-style cancellation without an expensive scan, run the cell below:
it should produce a cancelled-tool answer without starting model batches.
The ordinary earlier BRAPH cells already exercise **show 20**, **save
all**, and the deliberately noninteractive **show up to 100** paths.


In [ ]:
# Optional: the fact_finding intent always offers a plan when items exist.
# This tests a GUI Cancel choice; it is not included in matched coverage.
RUN_CANCEL_REGRESSION = True

if RUN_CANCEL_REGRESSION:
    store, conversation, session = create_conversation(project_dir)
    try:
        check_intent(
            session,
            "fact_finding",
            "List the programming languages in tiunn26braph2.pdf.",
            confirm_callback=confirm_gui_cancel,
        )
    finally:
        session.close()


### G. Coverage summary

Run this cell after the groups above. A `MATCH` means the **saved**
intent equals the intended intent and the saved tool name equals the
tool defined in `literature.json`. `INCOMPLETE` means routing matched, but the tool did not produce its expected result metadata. `SKIPPED` means the opt-in test was
not executed. If a route mismatches, inspect that question's trace and
revise the example or investigate the router; do not force the intent.


In [ ]:
coverage = show_intent_coverage()

# Full selection-path coverage also requires testing Continue and Cancel
# branches on appropriate non-network tool plans. The automated suite
# above focuses on the ordinary and Quick paths.
